# **Método proposto**

## **Descrição dos dados**

In [1]:
# Bibliotecas

from pathlib import Path # Para trabalhar com caminhos de arquivos e pastas
import re # Para expressões regulares
import pandas as pd
import numpy as np
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.base import clone

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

In [2]:

# importar toda a pasta de dados
Pasta_dados = Path("gait-in-neurodegenerative-disease-database-1.0.0")

# Verificar se foi importado corretamente
print(Pasta_dados.exists()) 

True


In [ ]:
# Precisamos separar entre HD e CO
def ordenar_numeros(caminho):

    # vai procurar o numero sem a extensão
    numero = re.search(r"\d+", caminho.stem) 
    # vai retornar apenas o número inteiro 
    return int(numero.group()) if numero else 0

hunt_data = sorted(
    # procura dentro da pasta de dados os arquivos que começam com 'hunt' e terminam com '.ts'
    Pasta_dados.rglob("hunt*.ts"),
    # Vamos ordenar com base na função
    key = ordenar_numeros
)

ctrl_data = sorted(
    # procura dentro da pasta de dados os arquivos que começam com 'control' e terminam com '.ts'
    Pasta_dados.rglob("control*.ts"),
    # Vamos ordenar com base na função
    key = ordenar_numeros
)

print("Arquivos Huntington:", len(hunt_data))
print("Arquivos Controle:", len(ctrl_data))

Arquivos Huntington: 20
Arquivos Controle: 16


In [4]:
COLUNAS_COMPLETAS = [
    "Elapsed Time (sec)",
    "Left Stride Interval (sec)",
    "Right Stride Interval (sec)",
    "Left Swing Interval (sec)",
    "Right Swing Interval (sec)",
    "Left Swing Interval (% of stride)",
    "Right Swing Interval (% of stride)",
    "Left Stance Interval (sec)",
    "Right Stance Interval (sec)",
    "Left Stance Interval (% of stride)",
    "Right Stance Interval (% of stride)",
    "Double Support Interval (sec)",
    "Double Support Interval (% of stride)"
]

COLUNAS_SINAIS = [
    "Left Stride Interval (sec)",
    "Right Stride Interval (sec)",
    "Left Swing Interval (sec)",
    "Right Swing Interval (sec)",
    "Left Stance Interval (sec)",
    "Right Stance Interval (sec)",
    "Double Support Interval (sec)"
]

In [5]:
# Carregar arquivos para cada paciente 
def carregar_arquivo_ts(caminho):
    df = pd.read_csv(
        caminho, 
        sep=r"\s+",
        header=None,
        names=COLUNAS_COMPLETAS
    )
    return df

## **Processamento dos dados**

In [6]:
# Vamos remover os primeiros 20 segundos de marcha de todos os pacientes em seus arquivos
def remover_20_segundos(df):
    df_sem_inicio = df[df['Elapsed Time (sec)'] > 20].copy()
    return df_sem_inicio

In [7]:
# Vamos aplicar filtro da mediana nas colunas usadas no artigo 

def filtro_mediana(serie):
    mediana = serie.median()
    desvio_padrao = serie.std()

    limite_inferior = mediana - 3 * desvio_padrao
    limite_superior = mediana + 3 * desvio_padrao

    outliers = (serie > limite_superior) | (serie < limite_inferior)


    serie_filtrada = serie.copy()
    serie_filtrada[outliers] = mediana
    return serie_filtrada, outliers.sum()


## **Extração das características (CV e Alpha)**

In [ ]:
# função para calcular o coeficiente de variância
def calcular_cv(serie):
    media = serie.mean()
    desvio = serie.std()

    CV = 100 * (desvio/media)
    return CV

In [9]:
def calcular_alpha(serie, n_min=10, n_max=20):
    # transformar a série em array
    x = np.array(serie.dropna(), dtype=float)

    # Se a série for muito pequena, não dá para calcular DFA
    if len(x) < n_max * 2:
        return np.nan

    # centralizar a série, além do seu comportamento médio
    x_centralizado = x - np.mean(x)

    # integrar a série
    y = np.cumsum(x_centralizado)

    # guardar tamanho de janelas e flutuações
    tamanho_janela = []
    flutuacoes = []

    for n in range(n_min, n_max + 1):

        n_segmentos = len(y) // n

        # Proteção computacional
        if n_segmentos < 2:
            continue

        # corta a série para caber em blocos completos
        y_cortado = y[:n_segmentos * n]

        # divide a série integrada em blocos de tamanho n
        segmentos = y_cortado.reshape(n_segmentos, n)

        # guarda a flutuação de cada segmento
        rms_segmentos = []

        for segmento in segmentos:
            # eixo de tempo dentro da janela
            tempo = np.arange(n)

            # ajusta uma reta no segmento
            coeficientes = np.polyfit(tempo, segmento, deg=1)
            tendencia = np.polyval(coeficientes, tempo)

            # remove a tendência local
            segmento_sem_tendencia = segmento - tendencia

            # calcula a flutuação RMS
            rms = np.sqrt(np.mean(segmento_sem_tendencia ** 2))
            rms_segmentos.append(rms)

        # calcula F(n), a flutuação média para esse tamanho de janela
        F_n = np.sqrt(np.mean(np.array(rms_segmentos) ** 2))

        tamanho_janela.append(n)
        flutuacoes.append(F_n)

    if len(flutuacoes) < 2:
        return np.nan

    # transforma em log para calcular a inclinação da reta log-log
    log_n = np.log(tamanho_janela)
    log_F = np.log(flutuacoes)

    # ajusta uma reta no gráfico log-log
    coeficientes = np.polyfit(log_n, log_F, deg=1)

    # alpha é a inclinação da reta
    alpha = coeficientes[0]

    return alpha

In [10]:
def processar_pacientes(caminho_arquivo, grupo, rotulo):
    df = carregar_arquivo_ts(caminho_arquivo)
    df = remover_20_segundos(df)

    total_outliers = {}

    resultados = []

    for col in COLUNAS_SINAIS:
        # Aplica filtro da mediana
        serie_filtrada, qntd_outliers = filtro_mediana(df[col])

        # Substitui a coluna original pela versão filtrada
        df[col] = serie_filtrada

        # Guarda quantidade de outliers encontrados
        total_outliers[col] = qntd_outliers

        # Calcula o CV dessa série filtrada
        cv = calcular_cv(serie_filtrada)
        alpha = calcular_alpha(serie_filtrada)

        # Guarda o resultado
        resultados.append({
            "arquivo": caminho_arquivo.name,
            "grupo": grupo,
            "rotulo": rotulo,
            "sinal": col,
            "cv": cv,
            "alpha": alpha,
            "outliers": qntd_outliers
        })


    return df, total_outliers, resultados

In [11]:
todos_resultados = []

for arquivo in hunt_data:
    df_processado, outliers, resultados = processar_pacientes(
        arquivo,
        grupo="HD",
        rotulo=1
    )

    todos_resultados.extend(resultados)


for arquivo in ctrl_data:
    df_processado, outliers, resultados = processar_pacientes(
        arquivo,
        grupo="CO",
        rotulo=0
    )

    todos_resultados.extend(resultados)


df = pd.DataFrame(todos_resultados)

df

,arquivo,grupo,rotulo,sinal,cv,alpha,outliers
0,hunt1.ts,HD,1,Left Stride Interval (sec),5.001032,0.485524,2
1,hunt1.ts,HD,1,Right Stride Interval (sec),5.018151,0.489427,5
2,hunt1.ts,HD,1,Left Swing Interval (sec),8.574813,0.406275,7
3,hunt1.ts,HD,1,Right Swing Interval (sec),7.656725,0.551005,4
4,hunt1.ts,HD,1,Left Stance Interval (sec),4.903443,0.632665,2
...,...,...,...,...,...,...,...
247,control16.ts,CO,0,Left Swing Interval (sec),4.887313,0.837613,5
248,control16.ts,CO,0,Right Swing Interval (sec),4.851449,0.678821,7
249,control16.ts,CO,0,Left Stance Interval (sec),4.995338,0.932428,4
250,control16.ts,CO,0,Right Stance Interval (sec),4.538913,0.931484,6


In [12]:
print(df.shape)
print(df["grupo"].value_counts())
print(df["sinal"].value_counts())
print(df[["cv", "alpha"]].isna().sum())

(252, 7)
grupo
HD    140
CO    112
Name: count, dtype: int64
sinal
Left Stride Interval (sec)       36
Right Stride Interval (sec)      36
Left Swing Interval (sec)        36
Right Swing Interval (sec)       36
Left Stance Interval (sec)       36
Right Stance Interval (sec)      36
Double Support Interval (sec)    36
Name: count, dtype: int64
cv       0
alpha    0
dtype: int64


In [13]:
df_features = df.copy()
df_features.head()


,arquivo,grupo,rotulo,sinal,cv,alpha,outliers
0,hunt1.ts,HD,1,Left Stride Interval (sec),5.001032,0.485524,2
1,hunt1.ts,HD,1,Right Stride Interval (sec),5.018151,0.489427,5
2,hunt1.ts,HD,1,Left Swing Interval (sec),8.574813,0.406275,7
3,hunt1.ts,HD,1,Right Swing Interval (sec),7.656725,0.551005,4
4,hunt1.ts,HD,1,Left Stance Interval (sec),4.903443,0.632665,2


In [14]:
df_features.shape

(252, 7)

In [15]:
df_features['grupo'].value_counts()

grupo
HD    140
CO    112
Name: count, dtype: int64

In [16]:
df_features["sinal"].value_counts()

sinal
Left Stride Interval (sec)       36
Right Stride Interval (sec)      36
Left Swing Interval (sec)        36
Right Swing Interval (sec)       36
Left Stance Interval (sec)       36
Right Stance Interval (sec)      36
Double Support Interval (sec)    36
Name: count, dtype: int64

In [17]:
df_features[["cv", "alpha"]].isna().sum()

cv       0
alpha    0
dtype: int64

In [18]:
dados_sinais = {}

for sinal in COLUNAS_SINAIS:
    df_sinal = df_features[df_features['sinal'] == sinal].copy()

    X = df_sinal[['cv', 'alpha']]
    y = df_sinal['rotulo']

    dados_sinais[sinal] = {
        'df_sinal': df_sinal,
        'X': X,
        'y': y
    }

    print("Sinal:", sinal)
    print("Formato de X:", X.shape)
    print("Formato de y:", y.shape)
    print("Classes:")
    print(y.value_counts())
    print("-" * 50)

Sinal: Left Stride Interval (sec)
Formato de X: (36, 2)
Formato de y: (36,)
Classes:
rotulo
1    20
0    16
Name: count, dtype: int64
--------------------------------------------------
Sinal: Right Stride Interval (sec)
Formato de X: (36, 2)
Formato de y: (36,)
Classes:
rotulo
1    20
0    16
Name: count, dtype: int64
--------------------------------------------------
Sinal: Left Swing Interval (sec)
Formato de X: (36, 2)
Formato de y: (36,)
Classes:
rotulo
1    20
0    16
Name: count, dtype: int64
--------------------------------------------------
Sinal: Right Swing Interval (sec)
Formato de X: (36, 2)
Formato de y: (36,)
Classes:
rotulo
1    20
0    16
Name: count, dtype: int64
--------------------------------------------------
Sinal: Left Stance Interval (sec)
Formato de X: (36, 2)
Formato de y: (36,)
Classes:
rotulo
1    20
0    16
Name: count, dtype: int64
--------------------------------------------------
Sinal: Right Stance Interval (sec)
Formato de X: (36, 2)
Formato de y: (36,

In [19]:
resumo_sinais = []

for sinal, dados in dados_sinais.items():
    X = dados['X']
    y = dados['y']

    resumo_sinais.append({
        'sinal': sinal,
        'linhas': X.shape[0],
        'caracteristicas': X.shape[1],
        'HD': (y==1).sum(),
        'CO': (y==0).sum()
    })

df_resumo_sinais = pd.DataFrame(resumo_sinais)

df_resumo_sinais

,sinal,linhas,caracteristicas,HD,CO
0,Left Stride Interval (sec),36,2,20,16
1,Right Stride Interval (sec),36,2,20,16
2,Left Swing Interval (sec),36,2,20,16
3,Right Swing Interval (sec),36,2,20,16
4,Left Stance Interval (sec),36,2,20,16
5,Right Stance Interval (sec),36,2,20,16
6,Double Support Interval (sec),36,2,20,16


## **Classificação**

In [20]:
MODELOS = {
    "SVM": SVC(kernel="linear"),
    "KNN": KNeighborsClassifier(n_neighbors=1, metric="euclidean"),
    "NB": GaussianNB(),
    "LDA": LinearDiscriminantAnalysis(),
    "DT": DecisionTreeClassifier(random_state=42)
}

In [21]:
def avaliar_modelo_loocv(X, y, modelo):
    loo = LeaveOneOut()

    y_reais = []
    y_previstos = []

    for treino_idx, teste_idx in loo.split(X):
        X_treino = X.iloc[treino_idx]
        X_teste = X.iloc[teste_idx]

        y_treino = y.iloc[treino_idx]
        y_teste = y.iloc[teste_idx]

        modelo_treino = clone(modelo)

        modelo_treino.fit(X_treino, y_treino)

        predicao = modelo_treino.predict(X_teste)

        y_reais.append(y_teste.iloc[0])
        y_previstos.append(predicao[0])

    acuracia = accuracy_score(y_reais, y_previstos)

    matriz = confusion_matrix(
        y_reais,
        y_previstos,
        labels=[1, 0]
    )

    return acuracia, matriz, y_reais, y_previstos

In [23]:
resultados_classificacao = []
matrizes_confusao = {}

for sinal, dados in dados_sinais.items():
    X = dados["X"]
    y = dados["y"]

    for nome_modelo, modelo in MODELOS.items():
        acuracia, matriz, y_reais, y_previstos = avaliar_modelo_loocv(
            X,
            y,
            modelo
        )

        resultados_classificacao.append({
            "sinal": sinal,
            "modelo": nome_modelo,
            "acuracia": acuracia,
            "acuracia_percentual": acuracia * 100
        })

        matrizes_confusao[(sinal, nome_modelo)] = matriz

In [24]:
df_resultados_classificacao = pd.DataFrame(resultados_classificacao)

df_resultados_classificacao

,sinal,modelo,acuracia,acuracia_percentual
0,Left Stride Interval (sec),SVM,0.916667,91.666667
1,Left Stride Interval (sec),KNN,0.888889,88.888889
2,Left Stride Interval (sec),NB,0.861111,86.111111
3,Left Stride Interval (sec),LDA,0.833333,83.333333
4,Left Stride Interval (sec),DT,0.861111,86.111111
5,Right Stride Interval (sec),SVM,0.944444,94.444444
6,Right Stride Interval (sec),KNN,0.944444,94.444444
7,Right Stride Interval (sec),NB,0.861111,86.111111
8,Right Stride Interval (sec),LDA,0.805556,80.555556
9,Right Stride Interval (sec),DT,0.972222,97.222222


In [25]:
tabela_acuracia = df_resultados_classificacao.pivot(
    index="modelo",
    columns="sinal",
    values="acuracia_percentual"
)

tabela_acuracia

sinal,Double Support Interval (sec),Left Stance Interval (sec),Left Stride Interval (sec),Left Swing Interval (sec),Right Stance Interval (sec),Right Stride Interval (sec),Right Swing Interval (sec)
modelo,,,,,,,
DT,94.444444,88.888889,86.111111,75.000000,100.000000,97.222222,83.333333
KNN,94.444444,86.111111,88.888889,77.777778,100.000000,94.444444,80.555556
LDA,58.333333,83.333333,83.333333,80.555556,80.555556,80.555556,83.333333
NB,94.444444,86.111111,86.111111,86.111111,91.666667,86.111111,91.666667
SVM,80.555556,91.666667,91.666667,83.333333,100.000000,94.444444,88.888889


In [26]:
df_resultados_classificacao.sort_values(
    by="acuracia_percentual",
    ascending=False
)

,sinal,modelo,acuracia,acuracia_percentual
29,Right Stance Interval (sec),DT,1.000000,100.000000
26,Right Stance Interval (sec),KNN,1.000000,100.000000
25,Right Stance Interval (sec),SVM,1.000000,100.000000
9,Right Stride Interval (sec),DT,0.972222,97.222222
34,Double Support Interval (sec),DT,0.944444,94.444444
32,Double Support Interval (sec),NB,0.944444,94.444444
31,Double Support Interval (sec),KNN,0.944444,94.444444
5,Right Stride Interval (sec),SVM,0.944444,94.444444
6,Right Stride Interval (sec),KNN,0.944444,94.444444
20,Left Stance Interval (sec),SVM,0.916667,91.666667


In [27]:
df_resultados_classificacao.sort_values(
    by="acuracia_percentual",
    ascending=True
)

,sinal,modelo,acuracia,acuracia_percentual
33,Double Support Interval (sec),LDA,0.583333,58.333333
14,Left Swing Interval (sec),DT,0.750000,75.000000
11,Left Swing Interval (sec),KNN,0.777778,77.777778
16,Right Swing Interval (sec),KNN,0.805556,80.555556
30,Double Support Interval (sec),SVM,0.805556,80.555556
28,Right Stance Interval (sec),LDA,0.805556,80.555556
8,Right Stride Interval (sec),LDA,0.805556,80.555556
13,Left Swing Interval (sec),LDA,0.805556,80.555556
19,Right Swing Interval (sec),DT,0.833333,83.333333
18,Right Swing Interval (sec),LDA,0.833333,83.333333


In [29]:
melhores_por_modelo = (
    df_resultados_classificacao
    .sort_values(by="acuracia_percentual", ascending=False)
    .drop_duplicates(subset="modelo")
    .reset_index(drop=True)
)

melhores_por_modelo

,sinal,modelo,acuracia,acuracia_percentual
0,Right Stance Interval (sec),DT,1.000000,100.000000
1,Right Stance Interval (sec),KNN,1.000000,100.000000
2,Right Stance Interval (sec),SVM,1.000000,100.000000
3,Double Support Interval (sec),NB,0.944444,94.444444
4,Right Swing Interval (sec),LDA,0.833333,83.333333


In [30]:
def mostrar_matriz_confusao(sinal, modelo):
    matriz = matrizes_confusao[(sinal, modelo)]

    matriz_df = pd.DataFrame(
        matriz,
        index=["Real HD", "Real CO"],
        columns=["Previsto HD", "Previsto CO"]
    )

    return matriz_df

In [31]:
for _, linha in melhores_por_modelo.iterrows():
    sinal = linha["sinal"]
    modelo = linha["modelo"]
    acuracia = linha["acuracia_percentual"]

    print("=" * 70)
    print("Modelo:", modelo)
    print("Melhor sinal:", sinal)
    print("Acurácia (%):", round(acuracia, 2))
    print("=" * 70)

    display(mostrar_matriz_confusao(sinal, modelo))

Modelo: DT
Melhor sinal: Right Stance Interval (sec)
Acurácia (%): 100.0


,Previsto HD,Previsto CO
Real HD,20,0
Real CO,0,16


Modelo: KNN
Melhor sinal: Right Stance Interval (sec)
Acurácia (%): 100.0


,Previsto HD,Previsto CO
Real HD,20,0
Real CO,0,16


Modelo: SVM
Melhor sinal: Right Stance Interval (sec)
Acurácia (%): 100.0


,Previsto HD,Previsto CO
Real HD,20,0
Real CO,0,16


Modelo: NB
Melhor sinal: Double Support Interval (sec)
Acurácia (%): 94.44


,Previsto HD,Previsto CO
Real HD,19,1
Real CO,1,15


Modelo: LDA
Melhor sinal: Right Swing Interval (sec)
Acurácia (%): 83.33


,Previsto HD,Previsto CO
Real HD,14,6
Real CO,0,16


In [32]:
resumo_matrizes = []

for _, linha in melhores_por_modelo.iterrows():
    sinal = linha["sinal"]
    modelo = linha["modelo"]
    acuracia = linha["acuracia_percentual"]

    matriz = matrizes_confusao[(sinal, modelo)]

    HD_previsto_HD = matriz[0, 0]
    HD_previsto_CO = matriz[0, 1]
    CO_previsto_HD = matriz[1, 0]
    CO_previsto_CO = matriz[1, 1]

    resumo_matrizes.append({
        "modelo": modelo,
        "melhor_sinal": sinal,
        "acuracia_percentual": round(acuracia, 2),
        "HD_previsto_HD": HD_previsto_HD,
        "HD_previsto_CO": HD_previsto_CO,
        "CO_previsto_HD": CO_previsto_HD,
        "CO_previsto_CO": CO_previsto_CO
    })

df_resumo_matrizes = pd.DataFrame(resumo_matrizes)

df_resumo_matrizes

,modelo,melhor_sinal,acuracia_percentual,HD_previsto_HD,HD_previsto_CO,CO_previsto_HD,CO_previsto_CO
0,DT,Right Stance Interval (sec),100.00,20,0,0,16
1,KNN,Right Stance Interval (sec),100.00,20,0,0,16
2,SVM,Right Stance Interval (sec),100.00,20,0,0,16
3,NB,Double Support Interval (sec),94.44,19,1,1,15
4,LDA,Right Swing Interval (sec),83.33,14,6,0,16
